# 🦅 PREDATOR Analytics v61.0-ELITE — CLUSTER DEPLOY ON COLAB (FIXED)

**Виправлення порівняно з попередньою версією:**
1. Docker запускається через `dockerd &` (Colab не має systemd)
2. `kubectl` встановлюється окремо
3. `zrok` шукається динамічно
4. Додано перевірки на кожному кроці

**Порядок:** Виконайте клітинки послідовно ⬇️

In [ ]:
%%bash
set -e

echo '═══════════════════════════════════════════════════════'
echo '🦅 PREDATOR v61.0-ELITE | [1/7] ІНФРАСТРУКТУРА'
echo '═══════════════════════════════════════════════════════'

# Docker
echo '📦 Встановлення Docker...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq ca-certificates curl gnupg > /dev/null 2>&1
install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | gpg --dearmor -o /etc/apt/keyrings/docker.gpg 2>/dev/null
chmod a+r /etc/apt/keyrings/docker.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] https://download.docker.com/linux/ubuntu $(. /etc/os-release && echo \"$VERSION_CODENAME\") stable" | tee /etc/apt/sources.list.d/docker.list > /dev/null
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq docker-ce docker-ce-cli containerd.io docker-buildx-plugin > /dev/null 2>&1

# ⚠️ FIX: Colab не має systemd — запускаємо dockerd вручну
echo '🐳 Запуск Docker daemon...'
dockerd --storage-driver=overlay2 > /tmp/dockerd.log 2>&1 &
for i in $(seq 1 30); do
    [ -S /var/run/docker.sock ] && break
    sleep 1
done
if ! docker ps > /dev/null 2>&1; then
    echo '❌ Docker не запустився!'; tail -10 /tmp/dockerd.log; exit 1
fi
echo "   ✅ Docker: $(docker version --format '{{.Server.Version}}')"

# K3d
echo '📦 Встановлення K3d...'
curl -s https://raw.githubusercontent.com/k3d-io/k3d/main/install.sh | bash 2>/dev/null
echo "   ✅ $(k3d version | head -1)"

# Kubectl
echo '📦 Встановлення kubectl...'
curl -sLO "https://dl.k8s.io/release/$(curl -sL https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl && rm -f kubectl
echo "   ✅ kubectl $(kubectl version --client -o json 2>/dev/null | python3 -c 'import sys,json;print(json.load(sys.stdin)["clientVersion"]["gitVersion"])' 2>/dev/null || echo 'installed')"

# Helm
echo '📦 Встановлення Helm...'
curl -fsSL https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash > /dev/null 2>&1
echo "   ✅ $(helm version --short)"

echo ''
echo '✅ ІНФРАСТРУКТУРА ГОТОВА!'

In [ ]:
%%bash
set -e

echo '═══════════════════════════════════════════════════════'
echo '🌐 [2/7] ВСТАНОВЛЕННЯ ZROK'
echo '═══════════════════════════════════════════════════════'

# Спробувати через apt
curl -sSLf https://get.openziti.io/install.bash | bash -s zrok 2>/dev/null || true

# Знайти бінарник
ZROK_BIN=$(which zrok 2>/dev/null || find /usr -name 'zrok' -type f 2>/dev/null | head -1)
if [ -z "$ZROK_BIN" ]; then
    echo '❌ zrok не знайдено!'
    exit 1
fi

echo "✅ zrok: $ZROK_BIN"
$ZROK_BIN version || true
echo "$ZROK_BIN" > /tmp/zrok_path.txt

In [ ]:
%%bash
set -e

echo '═══════════════════════════════════════════════════════'
echo '🔄 [3/7] КЛОНУВАННЯ РЕПОЗИТОРІЮ'
echo '═══════════════════════════════════════════════════════'

rm -rf /opt/predator
git clone --depth 1 https://github.com/dima1203oleg/predator-analytics.git /opt/predator
cd /opt/predator
echo "✅ $(git log --oneline -1)"

In [ ]:
%%bash
set -e

# Перевірка Docker
if ! docker ps > /dev/null 2>&1; then
    dockerd --storage-driver=overlay2 > /tmp/dockerd.log 2>&1 &
    for i in $(seq 1 30); do [ -S /var/run/docker.sock ] && break; sleep 1; done
fi

cd /opt/predator

echo '═══════════════════════════════════════════════════════'
echo '🔨 [4/7] ЗБІРКА КОНТЕЙНЕРІВ (~3-5 хв)'
echo '═══════════════════════════════════════════════════════'

for svc in frontend core-api graph-service ingestion-worker; do
    if [ "$svc" = "frontend" ]; then
        DF="apps/predator-analytics-ui/Dockerfile"
        CTX="."
    else
        DF="services/$svc/Dockerfile"
        CTX="services/$svc/"
    fi
    if [ -f "$DF" ]; then
        echo "🔨 Збірка $svc..."
        docker build -t predator/$svc:v61.0-ELITE -f $DF $CTX 2>&1 | tail -3
        echo "   ✅ $svc зібрано"
    else
        echo "   ⚠️ $DF не знайдено, пропускаю"
    fi
    echo ''
done

echo '📦 Образи:'
docker images | grep predator
echo ''
echo '✅ Збірка завершена!'

In [ ]:
%%bash
set -e

# Перевірка Docker
if ! docker ps > /dev/null 2>&1; then
    dockerd --storage-driver=overlay2 > /tmp/dockerd.log 2>&1 &
    for i in $(seq 1 30); do [ -S /var/run/docker.sock ] && break; sleep 1; done
fi

echo '═══════════════════════════════════════════════════════'
echo '🚀 [5/7] СТВОРЕННЯ K3D КЛАСТЕРА'
echo '═══════════════════════════════════════════════════════'

k3d cluster delete predator-cluster 2>/dev/null || true
sleep 3

k3d cluster create predator-cluster \
    -p "3030:30030@loadbalancer" \
    -p "8000:30080@loadbalancer" \
    -p "6443:6443@loadbalancer" \
    --k3s-arg "--disable=traefik@server:0" \
    --agents 1 \
    --wait

echo ''
kubectl cluster-info
kubectl get nodes

# Імпорт образів
echo ''
echo '📦 Імпорт образів...'
for img in $(docker images --format '{{.Repository}}:{{.Tag}}' | grep 'predator/'); do
    echo "   📥 $img"
    k3d image import "$img" -c predator-cluster 2>&1 | tail -1 || true
done

echo ''
echo '✅ Кластер готовий!'

In [ ]:
%%bash
set -e

cd /opt/predator

echo '═══════════════════════════════════════════════════════'
echo '⚙️ [6/7] ДЕПЛОЙ HELM CHART'
echo '═══════════════════════════════════════════════════════'

kubectl create namespace predator 2>/dev/null || true

helm upgrade --install predator ./deploy/helm/predator \
    --namespace predator \
    --set frontend.image.repository=predator/frontend \
    --set frontend.image.tag=v61.0-ELITE \
    --set frontend.image.pullPolicy=IfNotPresent \
    --set frontend.service.type=NodePort \
    --set frontend.service.nodePort=30030 \
    --set coreApi.image.repository=predator/core-api \
    --set coreApi.image.tag=v61.0-ELITE \
    --set coreApi.image.pullPolicy=IfNotPresent \
    --set coreApi.service.type=NodePort \
    --set coreApi.service.nodePort=30080 \
    --set global.domain=localhost \
    --wait --timeout 5m 2>&1 || echo '⚠️ Timeout, перевіряємо поди...'

echo ''
echo '📊 Поди:'
kubectl get pods -n predator -o wide
echo ''
echo '📊 Сервіси:'
kubectl get svc -n predator
echo ''
echo '✅ Helm chart розгорнуто!'

In [ ]:
import os, subprocess, re, time, threading

ZROK_TOKEN = '1eeje4um7yvA'

# Знайти zrok
zrok_bin = None
if os.path.exists('/tmp/zrok_path.txt'):
    zrok_bin = open('/tmp/zrok_path.txt').read().strip()
if not zrok_bin or not os.path.isfile(zrok_bin):
    for p in ['/usr/bin/zrok', '/usr/local/bin/zrok']:
        if os.path.isfile(p):
            zrok_bin = p
            break

print(f'🔑 zrok: {zrok_bin}')
subprocess.run(f'{zrok_bin} enable {ZROK_TOKEN}', shell=True, capture_output=True)

def run_share(name, port, results):
    proc = subprocess.Popen(
        [zrok_bin, 'share', 'public', f'http://localhost:{port}',
         '--headless', '--unique-name', name],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for _ in range(60):
        line = proc.stdout.readline()
        if not line: break
        m = re.search(r'https://[a-z0-9\-]+\.share\.zrok\.io', line)
        if m:
            results[name] = m.group(0)
            break
        time.sleep(0.5)
    results[f'{name}_proc'] = proc

results = {}
shares = [('predator-api', 6443), ('predator', 8000), ('predator-mirror', 3030)]

for name, port in shares:
    t = threading.Thread(target=run_share, args=(name, port, results), daemon=True)
    t.start()
    time.sleep(3)

time.sleep(15)

print('\n' + '═' * 60)
print('🦅 PREDATOR v61.0-ELITE | COLAB CLUSTER АКТИВНИЙ')
print('═' * 60)
for name, _ in shares:
    url = results.get(name, '⏳ очікування...')
    print(f'  {name}: {url}')
print('═' * 60)

In [ ]:
import time, datetime, subprocess
print('🦅 PREDATOR | Keep-Alive | Не закривайте!')
while True:
    pods = subprocess.run('kubectl get pods -n predator --no-headers 2>/dev/null | wc -l',
                         shell=True, capture_output=True, text=True)
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'\r⏱️ {ts} | Active | Pods: {pods.stdout.strip()}', end='', flush=True)
    time.sleep(60)